# DS201 - Lab 3: RNN cho phân loại văn bản và gán nhãn chuỗi

Chạy trên Kaggle với **Accelerator: GPU T4**. Thứ tự: cài đặt → kiểm tra dữ liệu → (tùy chọn) nạp checkpoint cũ → kiểm tra 1 batch → train từng bài.

> `/kaggle/working/` mất khi phiên kết thúc. Sau mỗi lần train, chạy cell cuối để tải `checkpoints.zip` về (hoặc *Save Version* để lưu output).

## 1. GPU

In [ ]:
import torch
print(torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: chưa bật GPU (Settings > Accelerator > GPU T4)")

## 2. Lấy code từ GitHub
Sửa `REPO_URL`. `LAB_SUBDIR` để trống thì notebook tự tìm thư mục chứa `assignment_1.py` (dù bạn đặt tên `BTTH3` hay `Lab3`).

In [ ]:
import os, glob

REPO_URL = "https://github.com/tnhyen/DS201-Deep-Learning.git"
REPO_DIR = "/kaggle/working/DS201-Deep-Learning"
LAB_SUBDIR = "BTTH3"   # thư mục Lab 3 trong repo; để trống = tự tìm

if "<username>" in REPO_URL:
    raise ValueError("Chưa sửa REPO_URL thành link GitHub thật.")

!rm -rf {REPO_DIR}
!git clone -q {REPO_URL} {REPO_DIR}

if LAB_SUBDIR:
    LAB_DIR = os.path.join(REPO_DIR, LAB_SUBDIR)
else:
    found = [os.path.dirname(p) for p in glob.glob(f"{REPO_DIR}/**/assignment_1.py", recursive=True)
             if "Sample" not in p]
    if len(found) != 1:
        raise RuntimeError(f"Tìm thấy {len(found)} thư mục chứa assignment_1.py: {found}. Hãy điền LAB_SUBDIR.")
    LAB_DIR = found[0]

if not os.path.exists(os.path.join(LAB_DIR, "assignment_1.py")):
    raise FileNotFoundError(f"Không thấy assignment_1.py trong {LAB_DIR}")

%cd {LAB_DIR}
!ls
!pip install -q seqeval   # cần cho Bài 3

## 3. Dữ liệu
- UIT-VSFC: 3 file JSON ở `/kaggle/working/UIT-VSFC/` (tải như cách bạn đã làm, hoặc add dataset rồi sửa `UIT_DIR`).
- PhoNER-COVID19: tự clone từ GitHub của VinAI, dùng bản **word**.

Thiếu file nào thì cell sẽ dừng với lỗi.

In [ ]:
UIT_DIR = "/kaggle/working/UIT-VSFC"
NER_DIR = "/kaggle/working/PhoNER_COVID19/data/word"

if not os.path.exists("/kaggle/working/PhoNER_COVID19"):
    !git clone -q https://github.com/VinAIResearch/PhoNER_COVID19.git /kaggle/working/PhoNER_COVID19

paths = [os.path.join(UIT_DIR, f"UIT-VSFC-{split}.json") for split in ["train", "dev", "test"]]
paths += [os.path.join(NER_DIR, f"{split}_word.json") for split in ["train", "dev", "test"]]

for path in paths:
    print("OK     " if os.path.exists(path) else "MISSING", path)

missing = [path for path in paths if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f"Missing files: {missing}")

## 4. (Tùy chọn) Nạp checkpoint từ phiên trước
Upload `checkpoints.zip` (đã tải về) thành Kaggle Dataset, add vào notebook rồi điền `OLD_CKPT_DIR`.

Mỗi bài chỉ được chạy tiếp khi có **đủ cả** `<model>_last.pt` (train tiếp) và `<model>_best.pt` (để test). Muốn train lại từ đầu thì thêm `--no_resume` vào lệnh chạy bài đó.

In [ ]:
import shutil

OLD_CKPT_DIR = ""   # ví dụ: "/kaggle/input/ds201-lab3-checkpoints"
CKPT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

def find_file(root, filename):
    found = glob.glob(f"{root}/**/{filename}", recursive=True)
    return found[0] if found else None

if OLD_CKPT_DIR:
    for name in ["lstm", "gru", "bilstm"]:
        files = {kind: find_file(OLD_CKPT_DIR, f"{name}_{kind}.pt") for kind in ["last", "best"]}
        if files["last"] and files["best"]:
            for src in files.values():
                shutil.copy2(src, CKPT_DIR)
            results = find_file(OLD_CKPT_DIR, f"{name}_results.json")
            if results:
                shutil.copy2(results, CKPT_DIR)
            print(f"{name}: đã nạp last + best -> sẽ chạy tiếp")
        elif files["last"] or files["best"]:
            print(f"{name}: THIẾU file ({files}) -> không nạp, bài này sẽ train từ đầu")
        else:
            print(f"{name}: chưa có checkpoint -> train từ đầu")

!ls -la {CKPT_DIR}

## 5. Kiểm tra 1 batch trước khi train
Đọc dữ liệu, kiểm tra shape / nhãn / padding, forward và tính loss cho cả 3 model. Mất vài giây.

In [ ]:
!python check_batch.py --uit_dir {UIT_DIR} --ner_dir {NER_DIR}

## 6. Bài 1 - LSTM trên UIT-VSFC

In [ ]:
!python assignment_1.py --data_dir {UIT_DIR} --checkpoint_dir {CKPT_DIR}

## 7. Bài 2 - GRU trên UIT-VSFC

In [ ]:
!python assignment_2.py --data_dir {UIT_DIR} --checkpoint_dir {CKPT_DIR}

## 8. Bài 3 - BiLSTM Encoder trên PhoNER-COVID19

In [ ]:
!python assignment_3.py --data_dir {NER_DIR} --checkpoint_dir {CKPT_DIR}

## 9. Tổng hợp kết quả và đóng gói checkpoint
Tải `/kaggle/working/checkpoints.zip` về ở mục Output, hoặc *Save Version* để giữ lại.

In [ ]:
import json

for path in sorted(glob.glob(f"{CKPT_DIR}/*_results.json")):
    print(json.dumps(json.load(open(path)), indent=2, ensure_ascii=False))

!cd /kaggle/working && zip -qr checkpoints.zip checkpoints && ls -lh checkpoints.zip